# Fase 4 — Fine-Tuning IndoBERT (Sentara)

Notebook ini dijalankan di **Google Colab (GPU)** karena GPU lokal (AMD Radeon Vega 7) tidak didukung PyTorch CUDA.

Alur: load data bersih (Fase 3) → focused random search (30% data) → pilih konfigurasi terbaik → full fine-tuning → 5-fold CV → evaluasi test set → simpan best model + log.

Metrik utama: **macro F1** (target ≥ 0.85). Class weight **wajib** (imbalance kelas > 15%).

Logika inti ada di `src/modeling/` — notebook hanya orkestrasi.

## 1. Setup environment
Clone repo (atau mount Google Drive), lalu install dependensi.

In [ ]:
# Opsi A: clone dari GitHub (ganti URL repo bila perlu)
# !git clone https://github.com/<user>/sistem.git
# %cd sistem

# Opsi B: mount Google Drive lalu cd ke folder proyek
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/SKRIPSI/sistem

!pip -q install "transformers>=4.40.0" "torch>=2.2.0" "datasets>=2.19.0" \
    "accelerate>=0.30.0" "scikit-learn>=1.4.0" "pandas>=2.2.0"

In [ ]:
import sys
from pathlib import Path

# Pastikan root proyek ada di sys.path agar `import src...` bekerja.
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
print('CUDA tersedia:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Load data bersih (output Fase 3)
`clean_train.csv` / `clean_validation.csv` / `clean_test.csv` (kolom: text, label, source).

In [ ]:
from src.modeling.data import load_clean_split, label_distribution, compute_class_weights
from src.modeling.config import LABEL2ID

train_df = load_clean_split('train')
val_df = load_clean_split('validation')
test_df = load_clean_split('test')

print('train/val/test:', len(train_df), len(val_df), len(test_df))
print('LABEL2ID:', LABEL2ID)
print('Distribusi train:', label_distribution(train_df))
print('Class weight:', [round(w, 3) for w in compute_class_weights(train_df)])

## 3. Focused random search (FR-4.6)
Baseline + 5–8 kombinasi acak, dilatih pada **30% training set** (stratified). Hasil → `outputs/reports/hyperparameter_log.csv`.

In [ ]:
from src.modeling.hyperparameter_search import run_focused_search
from src.modeling.config import SearchSpace

rows, best = run_focused_search(
    train_df=train_df,
    eval_df=val_df,
    space=SearchSpace(n_trials=6),
    seed=42,
)

import pandas as pd
display(pd.DataFrame(rows).sort_values('f1_macro', ascending=False))
print('Konfigurasi terbaik:', best)

## 4. Full fine-tuning dengan konfigurasi terbaik
Latih pada **full training set**, early stopping (patience=2) + LR scheduler linear+warmup. Best model (by macro F1) disimpan ke `models/best_model/`.

In [ ]:
from src.modeling.config import TrainingConfig, BEST_MODEL_DIR, ensure_output_dirs
from src.modeling.data import build_hf_dataset
from src.modeling.trainer import train_model, export_loss_history_csv
from src.preprocessing.tokenizer_wrapper import IndoBERTTokenizerWrapper

ensure_output_dirs()

# Konfigurasi final dari hasil search (sesuaikan jika perlu).
final_cfg = TrainingConfig(
    learning_rate=best['learning_rate'],
    batch_size=int(best['batch_size']),
    num_epochs=int(best['num_epochs']),
    run_name='final',
)

tokenizer = IndoBERTTokenizerWrapper().tokenizer
class_weights = compute_class_weights(train_df)
train_ds = build_hf_dataset(train_df, tokenizer=tokenizer)
val_ds = build_hf_dataset(val_df, tokenizer=tokenizer)

trainer, val_metrics = train_model(
    final_cfg,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    class_weights=class_weights,
    output_dir=BEST_MODEL_DIR,
    tokenizer=tokenizer,
    save_model=True,
)
print('Validation metrics:', val_metrics)

# Ekspor loss per epoch untuk learning curve Fase 5 (FR-5.6).
log_path = export_loss_history_csv(trainer)
print('Training log ->', log_path)

## 5. 5-Fold Stratified Cross-Validation (FR-4.7)
Validasi stabilitas konfigurasi final pada full training set → `outputs/reports/cross_validation_report.json`.

In [ ]:
from src.modeling.cross_validation import run_cross_validation

cv_report = run_cross_validation(train_df, config=final_cfg, n_splits=5, seed=42)
print('Mean F1: {f1_macro_mean} +/- {f1_macro_std}'.format(**cv_report))

## 6. Evaluasi test set & simpan metrik final
Evaluasi best model pada test set (10%) → `outputs/reports/phase4_final_metrics.json`. Gate Fase 4 menuntut macro F1 ≥ 0.85.

In [ ]:
import json
from src.modeling.config import FINAL_METRICS_JSON

test_ds = build_hf_dataset(test_df, tokenizer=tokenizer)
test_metrics = trainer.evaluate(test_ds)

final = {
    'accuracy': round(float(test_metrics['eval_accuracy']), 4),
    'precision_macro': round(float(test_metrics['eval_precision_macro']), 4),
    'recall_macro': round(float(test_metrics['eval_recall_macro']), 4),
    'f1_macro': round(float(test_metrics['eval_f1_macro']), 4),
    'config': final_cfg.as_dict(),
}
FINAL_METRICS_JSON.write_text(json.dumps(final, indent=2, ensure_ascii=False), encoding='utf-8')
print(json.dumps(final, indent=2, ensure_ascii=False))

## 7. Unduh artefak
Unduh `models/best_model/` + `outputs/reports/` ke lokal, lalu jalankan `python -m scripts.validate_phase4_gate` untuk verifikasi gate.

In [ ]:
# Arsipkan untuk diunduh (jika tidak via Drive)
# !zip -r best_model.zip models/best_model outputs/reports outputs/logs
# from google.colab import files
# files.download('best_model.zip')